- target feature로 Bad == 파산 or 부도 만 측정
- 상장폐지 이후는 모두 무시
- 시계열
- dataset_labelled.csv 사용

In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score

### 1. Raw dataset preview

In [2]:
df = pd.read_csv('dataset_labelled.csv')

print(df.shape)
print(df.head())
print(df.columns.to_list())

(96161, 265)
    Symbol  Name   결산월   회계년      주기       총자산(천원)   총자산(평균)(천원)  유동자산(천원)  \
0  A000010  조흥은행  12.0  1999  Annual  4.693918e+10  4.674953e+10       NaN   
1  A000010  조흥은행  12.0  2000  Annual  5.113869e+10  4.903894e+10       NaN   
2  A000010  조흥은행  12.0  2001  Annual  5.661012e+10  5.387440e+10       NaN   
3  A000010  조흥은행  12.0  2002  Annual  6.727043e+10  6.194027e+10       NaN   
4  A000010  조흥은행  12.0  2003  Annual  6.060160e+10  6.393601e+10       NaN   

   당좌자산(천원)  현금및현금성자산(천원)  ...  현금흐름(연율화)(천원)  현금흐름(직전4분기)(천원)  현금흐름(보통)(천원)  \
0       NaN  1.210399e+09  ...   -746322000.0              NaN  -746322000.0   
1       NaN  9.738450e+08  ...     99359000.0              NaN    99359000.0   
2       NaN  1.346084e+09  ...    710634000.0              NaN   710634000.0   
3       NaN  1.174976e+09  ...   -395695000.0              NaN  -395695000.0   
4       NaN  9.181470e+08  ...   -750930000.0              NaN  -750930000.0   

   현금흐름(보통,연율화)(천원)  현금흐름(보통,직전4분기)(천

### 2. 전처리

In [3]:
# 파생 지표 계산에 필요한 원시 컬럼들 (출력 제외 대상이지만 구성 요소는 체크해야 함)
required_columns = {
    "차입금": ["단기차입금(천원)", "장기차입금(천원)"],
    "운전자본": ["유동자산(천원)", "유동부채(천원)"],
    "총비용": ["매출액(천원)", "영업이익(천원)"],
    "EBITDA": ["영업이익(천원)", "유무형자산상각비(천원)...197"],
}

# 직접 결측치를 계산할 메트릭 목록
direct_metrics = [
    "총자산(천원)", "유형자산(천원)", "유동자산(천원)", "현금및현금성자산(천원)",
    "총부채(천원)", "유동부채(천원)", "총자본(천원)", "자본금(천원)", "*유보액(천원)",
    "매출액(천원)", "영업이익(천원)", "금융원가(비영업)(천원)", "당기순이익(천원)",
    "이익잉여금(천원)", "현금흐름(천원)", "영업활동으로인한현금흐름(천원)", "당기순손익(법인세차감전순손익)(천원)"
]

# 파생 지표 결과 그 자체는 결측치 계산에서 제외
exclude_metrics = {"EBITDA", "운전자본", "총비용", "차입금"}

# 결측치 결과 저장용 딕셔너리
missing = {}

# 직접 존재하는 항목들에 대해 결측치 계산
for metric in direct_metrics:
    if metric in df.columns:
        missing[metric] = df[metric].isna()
    else:
        missing[metric] = pd.Series([True] * len(df), index=df.index)

# 파생 지표를 구성하는 원시 컬럼들에 대해서만 결측치 계산 추가
for metric, cols in required_columns.items():
    for col in cols:
        if col not in missing:  # 중복 방지
            if col in df.columns:
                missing[col] = df[col].isna()
            else:
                missing[col] = pd.Series([True] * len(df), index=df.index)

# 결측치 여부 테이블 생성
missing_df = pd.DataFrame(missing)

# 결측치 여부 출력
print("결측치 여부 요약:")
display(missing_df)

# 결측치가 있는 지표만 요약
missing_summary = missing_df.sum().sort_values(ascending=False)
print("\n결측치가 많은 순으로 정렬:")
display(missing_summary[missing_summary > 0])

결측치 여부 요약:


,총자산(천원),유형자산(천원),유동자산(천원),현금및현금성자산(천원),총부채(천원),유동부채(천원),총자본(천원),자본금(천원),*유보액(천원),매출액(천원),영업이익(천원),금융원가(비영업)(천원),당기순이익(천원),이익잉여금(천원),현금흐름(천원),영업활동으로인한현금흐름(천원),당기순손익(법인세차감전순손익)(천원),단기차입금(천원),장기차입금(천원),유무형자산상각비(천원)...197
0,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False
1,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False
2,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False
3,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False
4,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96156,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
96157,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
96158,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
96159,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False



결측치가 많은 순으로 정렬:


유동자산(천원)        6448
단기차입금(천원)       6448
장기차입금(천원)       6448
유동부채(천원)        6448
*유보액(천원)          32
현금및현금성자산(천원)      11
총자산(천원)           11
유형자산(천원)          11
자본금(천원)           11
이익잉여금(천원)         11
총자본(천원)           11
총부채(천원)           11
dtype: int64

In [4]:
# 유동자산 구성 항목
current_asset_components = [
    "현금및현금성자산(천원)", "단기금융상품(천원)", "단기유가증권(천원)", "단기대여금(천원)",
    "매출채권(천원)", "단기미청구공사(천원)", "재고자산(천원)", "당기법인세자산(천원)", "매각예정비유동자산(천원)"
]

# 유동부채 구성 항목
current_liability_components = [
    "매입채무(천원)", "단기금융부채(천원)", "단기사채(천원)", "단기차입금(천원)",
    "유동성장기부채(천원)", "단기충당부채(천원)", "당기법인세부채(천원)", "매각예정처분자산집단부채(천원)"
]

# 유동자산(천원)의 결측치인 행만 계산하여 채우기
missing_rows_assets = df["유동자산(천원)"].isna()
if missing_rows_assets.any():
    asset_sums = df.loc[missing_rows_assets, current_asset_components].fillna(0).sum(axis=1)
    df.loc[missing_rows_assets, "유동자산(천원)"] = asset_sums

# 유동부채(천원)의 결측치인 행만 계산하여 채우기
missing_rows_liabilities = df["유동부채(천원)"].isna()
if missing_rows_liabilities.any():
    liability_sums = df.loc[missing_rows_liabilities, current_liability_components].fillna(0).sum(axis=1)
    df.loc[missing_rows_liabilities, "유동부채(천원)"] = liability_sums

### 2. 전처리

- Feature Selection

In [5]:
df["연도"] = df["회계년"].astype(int)
df["상장폐지년도"] = pd.to_datetime(df["상장폐지일자"], errors="coerce").dt.year

# 파생변수 생성
df['금융비용/부채'] = df['이자비용(비영업)(천원)'] / df['총부채(천원)']

total_cost = df['매출원가(천원)'] + df['판매비와관리비(천원)']
df['금융비용/총비용'] = df['이자비용(비영업)(천원)'] / total_cost

df['설비투자효율'] = df['매출액(천원)'] / df['유형자산(천원)']

df['현금흐름/총자본'] = df['영업활동으로인한현금흐름(천원)'] / df['총자본(천원)']

df['영업이익/총자본'] = df['영업이익(천원)'] / df['총자본(천원)']

df['유보이익/총자본'] = df['이익잉여금(천원)'] / df['총자본(천원)']

df['자기자본비율'] = df['총자본(천원)'] / df['총자산(천원)']

df['자본금회전율'] = df['매출액(천원)'] / df['자본금(천원)']

total_loans = df['단기차입금(천원)'] + df['장기차입금(천원)']
df['차입금의존도'] = total_loans / df['총자산(천원)']

df['총부채회전율'] = df['매출액(천원)'] / df['총부채(천원)']

df['총자본투자효율'] = df['영업이익(천원)'] / df['총자산(천원)']

df['총자본회전율'] = df['매출액(천원)'] / df['총자산(천원)']

df['총순자산수익률'] = df['당기순이익(천원)'] / df['총자본(천원)']

df['현금성자산비율'] = df['현금및현금성자산(천원)'] / df['총자산(천원)']


selected_input_features = [
    "Symbol",
    "연도",
    "상장폐지년도",
    '금융비용/부채',
    '금융비용/총비용',
    '설비투자효율',
    '현금흐름/총자본',
    '영업이익/총자본',
    '유보이익/총자본',
    '자기자본비율',
    '자본금회전율',
    '차입금의존도',
    '총부채회전율',
    '총자본투자효율',
    '총자본회전율',
    '총순자산수익률',
    '현금성자산비율'
]


target_features = [
    'Bad'
]

In [6]:
# 단기차입금 구성 항목
short_term_debt_components = [
    "단기금융부채(천원)", "단기사채(천원)", "유동성장기부채(천원)"
]

# 장기차입금 구성 항목
long_term_debt_components = [
    "장기금융부채(천원)", "사채(천원)", "금융리스부채(천원)"
]

# 단기차입금(천원)의 결측치인 행만 계산하여 채우기
if "단기차입금(천원)" in df.columns:
    missing_short_term = df["단기차입금(천원)"].isna()
    if missing_short_term.any():
        short_term_sums = df.loc[missing_short_term, short_term_debt_components].fillna(0).sum(axis=1)
        df.loc[missing_short_term, "단기차입금(천원)"] = short_term_sums

# 장기차입금(천원)의 결측치인 행만 계산하여 채우기
if "장기차입금(천원)" in df.columns:
    missing_long_term = df["장기차입금(천원)"].isna()
    if missing_long_term.any():
        long_term_sums = df.loc[missing_long_term, long_term_debt_components].fillna(0).sum(axis=1)
        df.loc[missing_long_term, "장기차입금(천원)"] = long_term_sums

In [7]:
# 차입금 = 단기차입금 + 장기차입금
if all(col in df.columns for col in ["단기차입금(천원)", "장기차입금(천원)"]):
    df["차입금(천원)"] = df["단기차입금(천원)"].fillna(0) + df["장기차입금(천원)"].fillna(0)
else:
    print("⚠️ '단기차입금(천원)' 또는 '장기차입금(천원)' 컬럼이 누락되었습니다.")

# 운전자본 = 유동자산 - 유동부채
if all(col in df.columns for col in ["유동자산(천원)", "유동부채(천원)"]):
    df["운전자본(천원)"] = df["유동자산(천원)"] - df["유동부채(천원)"]
else:
    print("⚠️ '유동자산(천원)' 또는 '유동부채(천원)' 컬럼이 누락되었습니다.")

# 총비용 = 매출액 - 영업이익
if all(col in df.columns for col in ["매출액(천원)", "영업이익(천원)"]):
    df["총비용(천원)"] = df["매출액(천원)"] - df["영업이익(천원)"]
else:
    print("⚠️ '매출액(천원)' 또는 '영업이익(천원)' 컬럼이 누락되었습니다.")

# EBITDA = 영업이익 + 유무형자산상각비
if all(col in df.columns for col in ["영업이익(천원)", "유무형자산상각비(천원)...197"]):
    df["EBITDA(천원)"] = df["영업이익(천원)"].fillna(0) + df["유무형자산상각비(천원)...197"].fillna(0)
else:
    print("⚠️ '영업이익(천원)' 또는 '유무형자산상각비(천원)' 컬럼이 누락되었습니다.")

In [8]:
# 파생 지표 계산에 필요한 원시 컬럼들 (출력 제외 대상이지만 구성 요소는 체크해야 함)
required_columns = {
    "차입금": ["단기차입금(천원)", "장기차입금(천원)"],
    "운전자본": ["유동자산(천원)", "유동부채(천원)"],
    "총비용": ["매출액(천원)", "영업이익(천원)"],
    "EBITDA": ["영업이익(천원)", "유무형자산상각비(천원)...197"],
}

# 직접 결측치를 계산할 메트릭 목록
direct_metrics = [
    "총자산(천원)", "유형자산(천원)", "유동자산(천원)", "현금및현금성자산(천원)",
    "총부채(천원)", "유동부채(천원)", "총자본(천원)", "자본금(천원)", "*유보액(천원)",
    "매출액(천원)", "영업이익(천원)", "금융원가(비영업)(천원)", "당기순이익(천원)",
    "이익잉여금(천원)", "현금흐름(천원)", "영업활동으로인한현금흐름(천원)", "당기순손익(법인세차감전순손익)(천원)"
]

# 파생 지표 결과 그 자체는 결측치 계산에서 제외
exclude_metrics = {"EBITDA", "운전자본", "총비용", "차입금"}

# 결측치 결과 저장용 딕셔너리
missing = {}

# 직접 존재하는 항목들에 대해 결측치 계산
for metric in direct_metrics:
    if metric in df.columns:
        missing[metric] = df[metric].isna()
    else:
        missing[metric] = pd.Series([True] * len(df), index=df.index)

# 파생 지표를 구성하는 원시 컬럼들에 대해서만 결측치 계산 추가
for metric, cols in required_columns.items():
    for col in cols:
        if col not in missing:  # 중복 방지
            if col in df.columns:
                missing[col] = df[col].isna()
            else:
                missing[col] = pd.Series([True] * len(df), index=df.index)

# 결측치 여부 테이블 생성
missing_df = pd.DataFrame(missing)

# 결측치 여부 출력
print("결측치 여부 요약:")
display(missing_df)

# 결측치가 있는 지표만 요약
missing_summary = missing_df.sum().sort_values(ascending=False)
print("\n결측치가 많은 순으로 정렬:")
display(missing_summary[missing_summary > 0])

결측치 여부 요약:


,총자산(천원),유형자산(천원),유동자산(천원),현금및현금성자산(천원),총부채(천원),유동부채(천원),총자본(천원),자본금(천원),*유보액(천원),매출액(천원),영업이익(천원),금융원가(비영업)(천원),당기순이익(천원),이익잉여금(천원),현금흐름(천원),영업활동으로인한현금흐름(천원),당기순손익(법인세차감전순손익)(천원),단기차입금(천원),장기차입금(천원),유무형자산상각비(천원)...197
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96156,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
96157,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
96158,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
96159,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False



결측치가 많은 순으로 정렬:


*유보액(천원)        32
총자산(천원)         11
현금및현금성자산(천원)    11
유형자산(천원)        11
이익잉여금(천원)       11
총부채(천원)         11
자본금(천원)         11
총자본(천원)         11
dtype: int64

- df 정리

In [9]:
selected_features = selected_input_features + target_features
df = df[selected_features].copy()

- inf & NA 처리

In [10]:
def process_missing_and_inf(group):
    group = group.replace([np.inf, -np.inf], np.nan)
    row_na_ratio = group.isna().mean(axis=1)

    cond_low = row_na_ratio < 0.10
    cond_mid = (row_na_ratio >= 0.10) & (row_na_ratio < 0.30)
    cond_high = row_na_ratio >= 0.30

    group = group[~cond_high]  # 제거
    group.loc[cond_mid, :] = group.loc[cond_mid, :].interpolate(method='linear', limit_direction='both')

    return group.dropna()



processed_groups = []

for symbol, group in df.groupby('Symbol'):
    processed_group = process_missing_and_inf(group)
    if not processed_group.empty:
        processed_groups.append(processed_group)

df = pd.concat(processed_groups, ignore_index=True)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17424\1615197886.py:10: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  group.loc[cond_mid, :] = group.loc[cond_mid, :].interpolate(method='linear', limit_direction='both')
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17424\1615197886.py:10: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  group.loc[cond_mid, :] = group.loc[cond_mid, :].interpolate(method='linear', limit_direction='both')
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17424\1615197886.py:10: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  group.loc[cond_mid, :] = group.loc[cond_mi

- 상장폐지 이후 row 제거

In [11]:
def remove_after_delisting(group):
    if group['상장폐지년도'].notna().any():
        first_delist_year = group['상장폐지년도'].dropna().iloc[0]
        return group[group['연도'] <= first_delist_year]
    return group

processed_groups = []

for symbol, group in df.groupby('Symbol'):
    processed_group = remove_after_delisting(group)
    if not processed_group.empty:
        processed_groups.append(processed_group)

df = pd.concat(processed_groups, ignore_index=True)

- 표준화

In [19]:
# 2. 스케일링 대상 컬럼만 선택
exclude_cols = ['Symbol', '연도', '상장폐지년도', 'Bad']
scaling_cols = [col for col in selected_input_features if col not in exclude_cols]

# 3. StandardScaler 적용
scaler = StandardScaler()
df_scaled_values = scaler.fit_transform(df[scaling_cols])

# 4. 스케일된 결과를 원본에 덮어씌움
df_scaled = df.copy()
df_scaled[scaling_cols] = df_scaled_values

- k개 년도 sliding window 생성

In [ ]:
def generate_sliding_windows(df, k):
    samples = []
    grouped = df.groupby('Symbol')

    for symbol, group in grouped:
        group = group.sort_values('연도')
        years = group['연도'].values
        features = group.drop(columns=['Symbol', '상장폐지년도']).values

        for i in range(len(group) - k):
            window_years = years[i:i + k + 1]
            if not np.all(np.diff(window_years) == 1):  # 연속 연도 아닌 경우 제거
                continue

            X_window = features[i:i + k] # k개 년도의 연속된 feature
            prediction_year = years[i + k] # 예측할 연도
            y_label = group.iloc[i + k]['Bad']

            samples.append((X_window, y_label, prediction_year))

    return samples

k = 3 # tuning parameter, k=5는 data가 너무 적어서
samples = generate_sliding_windows(df_scaled, k) # X_window, y_label, prediction_year

# prediction_year 기준으로 정렬
samples.sort(key=lambda x: x[2])

- y-label distriution

In [17]:
def check_label_distribution(dataset, name="Dataset"):
    # (X, y) 또는 (X, y, year) 모두 처리 가능
    labels = [y for item in dataset for i, y in enumerate(item) if i == 1]
    unique, counts = np.unique(labels, return_counts=True)

    print(f"\n{name} 샘플 수: {len(labels)}")
    for u, c in zip(unique, counts):
        print(f"- Label {u}: {c}개 ({c / len(labels):.2%})")


check_label_distribution(samples)


Dataset 샘플 수: 8069
- Label 0: 6718개 (83.26%)
- Label 1: 1351개 (16.74%)


- sample: X_window, y_label, prediction_year

In [18]:
print(f"1st sample example: \n{samples[0]}\n")

print(f"1st sample input: {samples[0][0]}")
print(f"1st sample label: {samples[0][1]}")
if samples[0][1] == 1:
    print(f"1st sample 상장폐지년도: {samples[0][2]}")
else:
    print(f"1st sample: 상장폐지 X")
print(f"1st sample 입력연도: {samples[0][2]-5}-{samples[0][2]-1}")
print(f"1st sample 예측연도: {samples[0][2]}")


print([s[2] for s in samples[:10]])  # 처음 10개 예측 연도 출력

1st sample example: 
(array([[ 1.99900000e+03,  5.74093154e-01,  1.50942653e-01,
        -1.17429106e-02,  1.66461260e-01,  5.00727179e-02,
        -3.29562513e-04, -4.51270005e-01, -9.48841963e-03,
         1.09031017e+00, -6.43330250e-01,  8.45074736e-02,
        -6.64847988e-01, -3.06786497e-02, -5.33383989e-01,
         0.00000000e+00],
       [ 2.00000000e+03,  1.20346648e+00,  3.48398066e-01,
        -1.17466840e-02,  3.15241687e-01,  2.54086683e-01,
         2.52838168e-02, -4.56398485e-01, -9.48937152e-03,
         1.33093378e+00, -6.45852489e-01,  2.04803203e-01,
        -6.71212866e-01,  1.53259278e-01, -7.91676582e-01,
         0.00000000e+00],
       [ 2.00100000e+03,  9.35890806e-01,  1.78965607e-01,
        -1.17406434e-02, -2.35692610e-01, -1.01989121e-01,
         8.66996396e-02, -6.02532827e-01, -9.48677229e-03,
         1.40788702e+00, -6.13997387e-01,  1.86464767e-01,
        -5.29937258e-01,  1.63808115e-01, -7.91676582e-01,
         0.00000000e+00]]), 1, np.int64(2

- 배열로 변환: TimeSeriesSplit 위함
- Test set 분리: 2021~2024년

In [20]:
X_window = np.array([s[0] for s in samples])
y_label = np.array([s[1] for s in samples])
prediction_year = np.array([s[2] for s in samples])


test_mask = (prediction_year >= 2021)
X_test = X_window[test_mask]
y_test = y_label[test_mask]

X_modeling = X_window[~test_mask]
y_modeling = y_label[~test_mask]

- Case sampling: y label ratio 1:1
- TimeSeriesSplit 도중 weight를 부여해서 비중을 1:1로 맞출 예정이예요.

In [21]:
def check_label_distribution_from_array(y, name=""):
    y = np.array(y)
    total = len(y)
    num_1 = np.sum(y == 1)
    num_0 = np.sum(y == 0)
    print(f"[{name}] Total: {total}, 1 비율: {num_1/total:.4f}, 1 개수: {num_1}, 0 개수: {num_0}")


check_label_distribution_from_array(y_modeling, "Modeling Set")
check_label_distribution_from_array(y_test, "Test Set")

[Modeling Set] Total: 7708, 1 비율: 0.1472, 1 개수: 1135, 0 개수: 6573
[Test Set] Total: 361, 1 비율: 0.5983, 1 개수: 216, 0 개수: 145


Test Set에서 covid19 때문인지 상장폐지 기업이 많네요.

### 3. Modeling

- MLP

In [22]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(input_size, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 1)
        )


    def forward(self, x):
        return self.fc(x)

# hyperparameters
n_splits = 5
batch_size = 64
lr = 1e-4
num_epochs = 100


input_size = X_modeling.shape[1] * X_modeling.shape[2]

Cross Validation: MLP
- accuracy와 함께 F1-socre를 사용했습니다. (unbalanced ratio data)

In [23]:
prediction_year_modeling = prediction_year[~(prediction_year >= 2021)]
assert len(prediction_year_modeling) == len(X_modeling)


tscv = TimeSeriesSplit(n_splits=n_splits)

print("==== TimeSeriesSplit MLP Training with Weighted Loss ====")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_modeling)):
    # Fold별 train/val 데이터 준비
    X_train = torch.tensor(X_modeling[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y_modeling[train_idx], dtype=torch.float32)
    X_val = torch.tensor(X_modeling[val_idx], dtype=torch.float32)
    y_val = torch.tensor(y_modeling[val_idx], dtype=torch.float32)

    # 연도 정보도 인덱싱 일관성 있게
    train_years = prediction_year_modeling[train_idx]
    val_years = prediction_year_modeling[val_idx]
    print(f"[Fold {fold+1}] 예측 연도 범위 (Train): {train_years.min()} ~ {train_years.max()}, (Val): {val_years.min()} ~ {val_years.max()}")

    # 가중치 계산 (positive label 보정)
    num_pos = (y_train == 1).sum().item()
    num_neg = (y_train == 0).sum().item()
    pos_weight = num_neg / num_pos if num_pos > 0 else 1.0
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    # 모델, 손실함수, 옵티마이저 정의
    model = MLP(input_size)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Dataloader
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=False)

    # Training
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb).squeeze()
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val).squeeze()
        val_probs = torch.sigmoid(val_logits).numpy()
        val_pred = (val_probs >= 0.5).astype(int)
        acc = accuracy_score(y_val.numpy(), val_pred)
        f1 = f1_score(y_val.numpy(), val_pred, zero_division=0)
        print(f"[Fold {fold+1}] Validation Accuracy: {acc:.4f}, F1 Score: {f1:.4f}\n")

==== TimeSeriesSplit MLP Training with Weighted Loss ====
[Fold 1] 예측 연도 범위 (Train): 2002 ~ 2004, (Val): 2004 ~ 2006
[Fold 1] Validation Accuracy: 0.0755, F1 Score: 0.1304

[Fold 2] 예측 연도 범위 (Train): 2002 ~ 2006, (Val): 2006 ~ 2008
[Fold 2] Validation Accuracy: 0.8995, F1 Score: 0.0153

[Fold 3] 예측 연도 범위 (Train): 2002 ~ 2008, (Val): 2008 ~ 2011
[Fold 3] Validation Accuracy: 0.7516, F1 Score: 0.1014

[Fold 4] 예측 연도 범위 (Train): 2002 ~ 2011, (Val): 2011 ~ 2015
[Fold 4] Validation Accuracy: 0.8474, F1 Score: 0.0200

[Fold 5] 예측 연도 범위 (Train): 2002 ~ 2015, (Val): 2015 ~ 2020
[Fold 5] Validation Accuracy: 0.7539, F1 Score: 0.1366



- GRU

In [24]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, bidirectional=False):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, _ = self.gru(x)
        # 마지막 타임스텝의 출력만 사용
        out = out[:, -1, :]  # shape: (batch, hidden_size)
        return self.fc(out)  # logits (no sigmoid)
    

    

# hyperparameters
n_splits = 5
batch_size = 64
lr = 1e-4
num_epochs = 30

input_dim = X_modeling.shape[2]  # 시계열 feature 수

In [25]:
# 연도 정렬 가정 하에 modeling dataset 추출
prediction_year_modeling = prediction_year[~(prediction_year >= 2021)]
assert len(prediction_year_modeling) == len(X_modeling)


tscv = TimeSeriesSplit(n_splits=n_splits)

print("==== TimeSeriesSplit GRU Training with Weighted Loss ====")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_modeling)):
    # Fold별 데이터 준비
    X_train = torch.tensor(X_modeling[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y_modeling[train_idx], dtype=torch.float32)
    X_val = torch.tensor(X_modeling[val_idx], dtype=torch.float32)
    y_val = torch.tensor(y_modeling[val_idx], dtype=torch.float32)

    # 연도 출력
    train_years = prediction_year_modeling[train_idx]
    val_years = prediction_year_modeling[val_idx]
    print(f"[Fold {fold+1}] 입력 연도 범위 (Train): {train_years.min()} ~ {train_years.max()}, (Val): {val_years.min()} ~ {val_years.max()}")

    # 클래스 불균형 보정
    num_pos = (y_train == 1).sum().item()
    num_neg = (y_train == 0).sum().item()
    pos_weight = num_neg / num_pos if num_pos > 0 else 1.0
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    # 모델 및 학습 설정
    model = GRUModel(input_size=X_modeling.shape[2])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=False)

    # 학습 루프
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            output = model(xb).squeeze()
            loss = criterion(output, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # 검증
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val).squeeze()
        val_probs = torch.sigmoid(val_logits).numpy()
        val_pred = (val_probs >= 0.5).astype(int)
        acc = accuracy_score(y_val.numpy(), val_pred)
        f1 = f1_score(y_val.numpy(), val_pred, zero_division=0)
        print(f"[Fold {fold+1}] Validation Accuracy: {acc:.4f}, F1 Score: {f1:.4f}\n")

==== TimeSeriesSplit GRU Training with Weighted Loss ====
[Fold 1] 입력 연도 범위 (Train): 2002 ~ 2004, (Val): 2004 ~ 2006
[Fold 1] Validation Accuracy: 0.0701, F1 Score: 0.1310

[Fold 2] 입력 연도 범위 (Train): 2002 ~ 2006, (Val): 2006 ~ 2008
[Fold 2] Validation Accuracy: 0.8995, F1 Score: 0.0000

[Fold 3] 입력 연도 범위 (Train): 2002 ~ 2008, (Val): 2008 ~ 2011
[Fold 3] Validation Accuracy: 0.8006, F1 Score: 0.0000

[Fold 4] 입력 연도 범위 (Train): 2002 ~ 2011, (Val): 2011 ~ 2015
[Fold 4] Validation Accuracy: 0.1542, F1 Score: 0.2672

[Fold 5] 입력 연도 범위 (Train): 2002 ~ 2015, (Val): 2015 ~ 2020
[Fold 5] Validation Accuracy: 0.2336, F1 Score: 0.3780



- TCN

In [26]:
class Chomp1d(nn.Module):
    """Conv1d의 padding 때문에 생기는 output 뒤쪽 값을 잘라내는 역할"""
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size]


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, kernel_size, stride=stride,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TCN(nn.Module):
    def __init__(self, input_size, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_ch = input_size if i == 0 else num_channels[i - 1]
            out_ch = num_channels[i]
            layers += [TemporalBlock(in_ch, out_ch, kernel_size, stride=1,
                                     dilation=dilation_size,
                                     padding=(kernel_size - 1) * dilation_size,
                                     dropout=dropout)]
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 1)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = x.permute(0, 2, 1)  # → (batch, input_dim, seq_len)
        y = self.tcn(x)         # → (batch, out_ch, seq_len)
        y = y[:, :, -1]         # 마지막 타임스텝만
        return self.fc(y)       # logits 출력

In [27]:
# 연도 정렬 가정 하에 modeling dataset 추출
prediction_year_modeling = prediction_year[~(prediction_year >= 2021)]
assert len(prediction_year_modeling) == len(X_modeling)


tscv = TimeSeriesSplit(n_splits=n_splits)

print("==== TimeSeriesSplit TCN Training with Weighted Loss ====")
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_modeling)):
    # Fold별 데이터
    X_train = torch.tensor(X_modeling[train_idx], dtype=torch.float32)  # (batch, seq_len, input_dim)
    y_train = torch.tensor(y_modeling[train_idx], dtype=torch.float32)
    X_val = torch.tensor(X_modeling[val_idx], dtype=torch.float32)
    y_val = torch.tensor(y_modeling[val_idx], dtype=torch.float32)

    train_years = prediction_year_modeling[train_idx]
    val_years = prediction_year_modeling[val_idx]
    print(f"[Fold {fold+1}] 입력 연도 범위 (Train): {train_years.min()} ~ {train_years.max()}, (Val): {val_years.min()} ~ {val_years.max()}")

    # 클래스 비율에 따른 가중치
    num_pos = (y_train == 1).sum().item()
    num_neg = (y_train == 0).sum().item()
    pos_weight = num_neg / num_pos if num_pos > 0 else 1.0
    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

    # TCN 모델 정의
    model = TCN(
        input_size=X_modeling.shape[2],   # input_dim = feature 수
        num_channels=[64, 64, 64],
        kernel_size=3,
        dropout=0.3
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=False)

    # 학습 루프
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb).squeeze()
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # 검증
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val).squeeze()
        val_probs = torch.sigmoid(val_logits).numpy()
        val_pred = (val_probs >= 0.5).astype(int)
        acc = accuracy_score(y_val.numpy(), val_pred)
        f1 = f1_score(y_val.numpy(), val_pred, zero_division=0)
        print(f"[Fold {fold+1}] Validation Accuracy: {acc:.4f}, F1 Score: {f1:.4f}\n")

==== TimeSeriesSplit TCN Training with Weighted Loss ====
[Fold 1] 입력 연도 범위 (Train): 2002 ~ 2004, (Val): 2004 ~ 2006
[Fold 1] Validation Accuracy: 0.9260, F1 Score: 0.1284

[Fold 2] 입력 연도 범위 (Train): 2002 ~ 2006, (Val): 2006 ~ 2008
[Fold 2] Validation Accuracy: 0.8964, F1 Score: 0.0148

[Fold 3] 입력 연도 범위 (Train): 2002 ~ 2008, (Val): 2008 ~ 2011
[Fold 3] Validation Accuracy: 0.2002, F1 Score: 0.3327

[Fold 4] 입력 연도 범위 (Train): 2002 ~ 2011, (Val): 2011 ~ 2015
[Fold 4] Validation Accuracy: 0.1573, F1 Score: 0.2679

[Fold 5] 입력 연도 범위 (Train): 2002 ~ 2015, (Val): 2015 ~ 2020
[Fold 5] Validation Accuracy: 0.2329, F1 Score: 0.3778



### 4. Test Set

In [28]:
def evaluate_model(model, X, y_true, model_name):
    model.eval()
    with torch.no_grad():
        logits = model(X).squeeze()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()
        acc = accuracy_score(y_true.numpy(), preds.numpy())
        f1 = f1_score(y_true.numpy(), preds.numpy(), zero_division=0)
        print(f"[{model_name}] Test Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")


# Tensor 변환
X_test_mlp = torch.tensor(X_test, dtype=torch.float32)             # (N, input_dim)
X_test_seq = torch.tensor(X_test, dtype=torch.float32)             # (N, seq_len, input_dim)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

model_mlp = MLP(X_modeling.shape[1] * X_modeling.shape[2])
model_gru= GRUModel(X_modeling.shape[2])
model_tcn = TCN(
    input_size=X_modeling.shape[2],   # input_dim = feature 수
    num_channels=[64, 64, 64],
    kernel_size=3,
    dropout=0.3
)

# 평가
evaluate_model(model_mlp, X_test_mlp, y_test_tensor, "MLP")
evaluate_model(model_gru, X_test_seq, y_test_tensor, "GRU")
evaluate_model(model_tcn, X_test_seq, y_test_tensor, "TCN")


[MLP] Test Accuracy: 0.5983, F1 Score: 0.7487
[GRU] Test Accuracy: 0.5983, F1 Score: 0.7487
[TCN] Test Accuracy: 0.5983, F1 Score: 0.7487
